# Notebook 01 - Preprocess Health Data

Upload `data/raw/labeled_health_data.csv` len Google Colab, sau do chay tung cell de tao `health_data_clean.csv` va `scaler.pkl`.

In [ ]:
# Cell 1: Import
import pandas as pd, numpy as np
from sklearn.preprocessing import MinMaxScaler
import joblib, matplotlib.pyplot as plt
print('Import OK')

In [ ]:
# Cell 2: Load va kiem tra
df = pd.read_csv('labeled_health_data.csv')
print(f'Tong mau: {len(df)}')
print(f'So subjects: {df["subject_id"].nunique()}')
print('\nPhan bo nhan:')
print(df['label'].value_counts())
print('\nThong ke:')
print(df[['heart_rate','spo2','temperature']].describe())

In [ ]:
# Cell 3: Lam sach
df = df.dropna()
df = df[(df['heart_rate'] > 30) & (df['heart_rate'] < 220)]
df = df[(df['spo2'] >= 50) & (df['spo2'] <= 100)]
df = df[(df['temperature'] > 30) & (df['temperature'] < 43)]
print(f'Sau lam sach: {len(df)} mau')

In [ ]:
# Cell 4: Feature engineering (rolling window PER SESSION)
df = df.sort_values(['subject_id','session','timestamp']).reset_index(drop=True)
df['hr_rolling_mean'] = df.groupby(['subject_id','session'])['heart_rate']\
                          .transform(lambda x: x.rolling(5,min_periods=1).mean())
df['hr_rolling_std']  = df.groupby(['subject_id','session'])['heart_rate']\
                          .transform(lambda x: x.rolling(5,min_periods=1).std().fillna(0))
df['spo2_change']     = df.groupby(['subject_id','session'])['spo2']\
                          .transform(lambda x: x.diff().fillna(0))
df['accel_mag']       = (df['accel_x']**2 + df['accel_y']**2 + df['accel_z']**2)**0.5
print('Feature engineering xong:', df.columns.tolist())

In [ ]:
# Cell 5: Chuan hoa va luu
features = ['heart_rate','spo2','temperature','hr_rolling_mean',
            'hr_rolling_std','spo2_change','accel_mag']
scaler = MinMaxScaler()
df_scaled = df.copy()
df_scaled[features] = scaler.fit_transform(df[features])
joblib.dump(scaler, 'scaler.pkl')
df_scaled.to_csv('health_data_clean.csv', index=False)
print('Hoan tat! Files da luu.')

In [ ]:
# Cell 6: Download ket qua tu Colab
try:
    from google.colab import files
    files.download('health_data_clean.csv')
    files.download('scaler.pkl')
except Exception as exc:
    print('Khong phai Colab hoac khong download duoc:', exc)